# yield

## 基本使用示例

In [4]:
def inner():
    print("  [inner] 准备执行 yield...")
    x = yield "给我数据"
    print(f"  [inner] x 拿到了: {x}")
    print(f"  [inner] x 的 id 是: {id(x)}")
    yield "处理完毕"

g = inner()
print("调用方: next(g) 启动")
result = next(g)
print(f"调用方: 收到吐出的值 -> {result!r}")
print("调用方: 此时 inner 冻结在 yield 处, x 还不存在")
print("调用方: 现在发送 42")
g.send(42)


调用方: next(g) 启动
  [inner] 准备执行 yield...
调用方: 收到吐出的值 -> '给我数据'
调用方: 此时 inner 冻结在 yield 处, x 还不存在
调用方: 现在发送 42
  [inner] x 拿到了: 42
  [inner] x 的 id 是: 140715632099032


'处理完毕'

 yield 作为生成器，目前主要有三个应用。

## 场景一：流式处理海量数据 📊
**业界痛点：** 你需要读取一个 50GB 的服务器日志文件，分析里面的报错信息。如果用普通函数一次性读进列表，服务器的内存会瞬间被撑爆（MemoryError）。  
**yield 的解法：** 流式读取（Streaming）。一行一行地读，处理完一行就立刻丢弃，内存占用永远只有一行文本的大小。

```python
def read_large_log(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "ERROR" in line:
                yield line  # 找到错误就吐出来，然后暂停

# 实际调用：内存占用极低，哪怕文件有 100GB
for error_log in read_large_log('server.log'):
    print(error_log)
```

## 场景二：生成无限序列 🔄
**业界痛点：** 你需要为数据库不断生成不重复的全局唯一 ID，或者在测试时生成无限的假数据。你不可能提前在内存里建一个包含无限个数字的列表。  
**yield 的解法：** 无限迭代器（Infinite Iterator）。利用 yield 配合 while 循环，按需生成。

```python
def unique_id_generator():
    user_id = 1000
    while True:
        yield f"USER_{user_id}"
        user_id += 1  # 状态会被冻结并保留

# 实际调用：每次 next() 都会安全地获得一个新ID
id_gen = unique_id_generator()
print(next(id_gen)) # 输出: USER_1000
print(next(id_gen)) # 输出: USER_1001
```

## 场景三：优雅的资源管理 (上下文管理器) 🔒
**业界痛点：** 连接数据库或操作文件后，很容易忘记手动关闭连接，导致资源泄露。  
**yield 的解法：** 利用 Python 官方库 contextlib。通过在 yield 的前后写上代码，可以自动生成 with 语句，在 yield 之前“获取资源”，在 yield 之后“清理资源”。

```python
from contextlib import contextmanager

@contextmanager
def database_connection():
    print("1. 打开数据库连接")
    conn = {"status": "connected"} # 模拟获取连接
    
    yield conn  # 暂停！把连接交给 with 语句去使用
    
    print("3. 自动关闭数据库连接") # 无论内部是否报错，这里一定会执行

# 实际调用：
with database_connection() as my_db:
    print(f"2. 正在使用: {my_db['status']}")
    # 出了 with 缩进块，立刻执行 yield 后面的清理代码
```


### 数据分批处理器

In [1]:
def batcher(iterable, size):
    res = []
    # 直接遍历数据，不需要管它的总长度，也不需要索引
    for item in iterable:
        res.append(item)
        if len(res) == size:  # 攒够了 size 个
            yield res         # 吐出去
            res = []          # 清空容器，准备下一批
            
    # 循环结束后，如果 res 里还有剩的（且不为空），再吐出去
    if res: 
        yield res

# 测试：
data = [1, 2, 3, 4, 5, 6, 7]
for batch in batcher(data, 3):
    print(batch)

# 预期输出：
# [1, 2, 3]
# [4, 5, 6]
# [7]  <-- 注意这里，剩下一个也要吐出来

[1, 2, 3]
[4, 5, 6]
[7]


### 性能计时器 (Timer)

In [2]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label="耗时"):
    # 1. 进门：记录开始时间
    start_time = time.time() 
    
    # 2. 交出控制权
    yield 
    
    # 3. 出门：当 with 内部代码执行完毕，从这里苏醒并计算
    end_time = time.time()
    print(f"[{label}] 耗时: {end_time - start_time:.4f} 秒")
    

# 测试：
with timer("睡觉任务"):
    print("开始睡觉...")
    time.sleep(1) # 模拟耗时操作：强制暂停1秒
    print("睡醒了！")

# 预期输出：
# 开始睡觉...
# 睡醒了！
# [睡觉任务] 耗时: 1.00x 秒

开始睡觉...
睡醒了！
[睡觉任务] 耗时: 1.0011 秒


## yield from

In [3]:
def flatten(items):
    for x in items:
        if isinstance(x, list):
            yield from flatten(x)   # 递归委托
        else:
            yield x

list(flatten([1, [2, [3, 4], 5], 6]))
# [1, 2, 3, 4, 5, 6]

[1, 2, 3, 4, 5, 6]